# Bedrock AgentCore Gateway를 사용하여 Smithy API를 MCP 도구로 변환하기

## 개요
Bedrock AgentCore Gateway를 사용하면 인프라나 호스팅을 관리하지 않고도 기존 Smithy API를 완전관리형 MCP 서버로 전환할 수 있습니다. Smithy 사양을 가져와 MCP 도구로 변환할 수 있습니다. 여기서는 Amazon S3의 Smithy 모델에서 MCP 도구를 생성하는 방법을 살펴봅니다. 그러면 에이전트가 Amazon S3를 조회하고 관련 질문에 답할 수 있습니다.

Gateway 워크플로에서는 에이전트를 외부 도구에 연결하기 위해 다음 단계를 수행합니다.
* **Gateway용 도구 생성** - Smithy 사양을 사용하여 도구를 정의합니다. 
* **Gateway 엔드포인트 생성** - 인바운드 인증을 사용하는 MCP 진입점 역할을 할 Gateway를 생성합니다.
* **Gateway에 대상 추가** - Gateway가 요청을 특정 도구로 라우팅하는 방식을 정의하는 Smithy 대상을 구성합니다. Smithy 파일에 포함된 모든 작업은 MCP 호환 도구가 되며 Gateway 엔드포인트 URL을 통해 제공됩니다. Smithy를 통해 Amazon S3 API를 호출할 수 있도록 AWS IAM을 사용하여 아웃바운드 권한 부여를 구성합니다.
* **에이전트 코드 업데이트** - 통합 MCP 인터페이스를 통해 구성된 모든 도구에 액세스하도록 에이전트를 Gateway 엔드포인트에 연결합니다.

![작동 방식](images/smithy-apis-gateway.png)

### 튜토리얼 세부 정보


| 정보                 | 세부 정보                                                 |
|:---------------------|:----------------------------------------------------------|
| 튜토리얼 유형        | 대화형                                                    |
| AgentCore 구성 요소  | AgentCore Gateway, AgentCore Identity                     |
| 에이전틱 프레임워크  | Strands Agents                                            |
| Gateway 대상 유형   | Smithy                                                    |
| 에이전트             | 인프라 에이전트                                           |
| 인바운드 인증 IdP   | Amazon Cognito                                            |
| 아웃바운드 인증     | IAM                                                       |
| LLM 모델             | Anthropic Claude Haiku 4.5, Amazon Nova Pro              |
| 튜토리얼 구성 요소  | AgentCore Gateway 생성 및 호출                           |
| 튜토리얼 분야       | 범분야                                                    |
| 예제 난이도         | 쉬움                                                      |
| 사용 SDK            | boto3                                                     |

튜토리얼의 첫 번째 부분에서는 몇 가지 AmazonCore Gateway 대상을 생성합니다.

### 튜토리얼 아키텍처
이 튜토리얼에서는 Amazon S3의 Smithy 사양에 정의된 작업을 MCP 도구로 변환하고 Bedrock AgentCore Gateway에서 호스팅합니다. 이를 통해 사용자는 자신의 AWS 계정에 있는 Amazon S3와 관련된 질문을 할 수 있습니다.

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음이 필요합니다.
* Python 3.10 이상을 사용하는 Jupyter Notebook
* uv
* AWS 자격 증명
* Amazon Cognito

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
import os

# SageMaker Notebook을 사용하지 않는 경우 AWS 자격 증명 설정
# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ["AWS_DEFAULT_REGION"] = os.environ.get("AWS_REGION", "us-east-1")

In [ ]:
import os
import sys

# 현재 스크립트의 디렉터리 확인
if "__file__" in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # __file__이 정의되지 않은 경우의 대체 경로(예: Jupyter)

# utils.py가 있는 디렉터리로 이동(한 수준 위)
utils_dir = os.path.abspath(os.path.join(current_dir, "../.."))

# sys.path에 추가
sys.path.insert(0, utils_dir)

# 이제 utils를 가져올 수 있음
import utils

In [ ]:
#### Gateway에서 수임할 IAM 역할 생성

agentcore_gateway_iam_role = utils.create_agentcore_gateway_role_s3_smithy("sample-lambdagateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role["Role"]["Arn"])

# Gateway 인바운드 권한 부여를 위한 Amazon Cognito 풀 생성

In [ ]:
# Cognito 사용자 풀 생성
import os
import boto3

REGION = os.environ["AWS_DEFAULT_REGION"]
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {"ScopeName": "gateway:read", "ScopeDescription": "Read access"},
    {"ScopeName": "gateway:write", "ScopeDescription": "Write access"},
]
scopeString = f"{RESOURCE_SERVER_ID}/gateway:read {RESOURCE_SERVER_ID}/gateway:write"

cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {user_pool_id}")

utils.get_or_create_resource_server(cognito, user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

client_id, client_secret = utils.get_or_create_m2m_client(cognito, user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID)
print(f"Client ID: {client_id}")

# 검색 URL 확인
cognito_discovery_url = f"https://cognito-idp.{REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"
print(cognito_discovery_url)

# Gateway 생성

In [ ]:
# CMK 없이 Cognito 권한 부여자를 사용하여 CreateGateway 호출. 이전 단계에서 생성한 Cognito 사용자 풀 사용
import boto3

gateway_client = boto3.client("bedrock-agentcore-control", region_name=os.environ["AWS_DEFAULT_REGION"])
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            client_id
        ],  # 클라이언트는 Cognito에 구성된 ClientId와 반드시 일치해야 함. 예: 7rfbikfsm51j2fpaggacgng84g
        "discoveryUrl": cognito_discovery_url,
    }
}
create_response = gateway_client.create_gateway(
    name="DemoS3Smithyv3",
    roleArn=agentcore_gateway_iam_role["Role"][
        "Arn"
    ],  # IAM 역할에는 Gateway 생성/나열/조회/삭제 권한이 있어야 함
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration=auth_config,
    description="AgentCore Gateway with Smithy target",
)
print(create_response)
# GatewayTarget 생성에 사용할 GatewayID 조회
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)

# Amazon S3 사양으로 Smithy 대상 생성

Amazon S3 API의 Smithy Gateway 대상을 생성합니다. Amazon S3 Smithy JSON 파일을 가져와 API를 MCP 도구로 변환합니다.

#### S3 Smithy API JSON 파일을 S3에 업로드

In [ ]:
# S3 클라이언트 생성
session = boto3.session.Session()
s3_client = session.client("s3")
sts_client = session.client("sts")

# AWS 계정 ID와 리전 조회
account_id = sts_client.get_caller_identity()["Account"]
region = session.region_name
# 파라미터 정의
bucket_name = f"agentcore-gateway-{account_id}-{region}"  # OpenAPI JSON 파일을 업로드할 S3 버킷
file_path = "smithy-specs/s3-apis.json"
object_key = "s3-apis.json"
# put_object를 사용하여 파일을 업로드하고 응답 확인
try:
    try:
        if region == "us-east-1":
            s3bucket = s3_client.create_bucket(Bucket=bucket_name)
        else:
            s3bucket = s3_client.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={"LocationConstraint": region},
            )
    except Exception as e:
        print(e)
    with open(file_path, "rb") as file_data:
        response = s3_client.put_object(Bucket=bucket_name, Key=object_key, Body=file_data)

    # 계정 ID와 리전을 사용하여 업로드된 객체의 ARN 구성
    smithy_s3_uri = f"s3://{bucket_name}/{object_key}"
    print(f"Uploaded object S3 URI: {smithy_s3_uri}")
except Exception as e:
    print(f"Error uploading file: {e}")

#### Gateway 대상 생성

In [ ]:
# Smithy 사양 파일의 S3 URI
smithy_s3_target_config = {"mcp": {"smithyModel": {"s3": {"uri": smithy_s3_uri}}}}

# IAM 자격 증명 공급자 구성
credential_config = {"credentialProviderType": "GATEWAY_IAM_ROLE"}

targetname = "DemoSmithytargetForS3"
response = gateway_client.create_gateway_target(
    gatewayIdentifier=gatewayID,
    name=targetname,
    description="Smithy Target with S3Uri using SDK",
    targetConfiguration=smithy_s3_target_config,
    credentialProviderConfigurations=[credential_config],
)

# 결함 보고에 사용할 요청 ID와 타임스탬프 출력. 문제나 결함을 보고할 때 함께 포함
response_metadata = response["ResponseMetadata"]
print(response_metadata)

# Strands Agent에서 Bedrock AgentCore Gateway 호출하기

Strands Agent는 Model Context Protocol(MCP) 사양을 구현하는 Bedrock AgentCore Gateway를 통해 AWS 도구와 원활하게 통합됩니다. 이 통합을 통해 AI 에이전트와 AWS 서비스가 안전하고 표준화된 방식으로 통신할 수 있습니다.

Bedrock AgentCore Gateway는 기본적으로 핵심 MCP API인 ListTools와 InvokeTools를 제공하는 프로토콜 호환 Gateway 역할을 합니다. 이러한 API를 통해 MCP 호환 클라이언트 또는 SDK는 사용 가능한 도구를 검색하고 안전하고 표준화된 방식으로 상호 작용할 수 있습니다. Strands Agent가 AWS 서비스에 액세스해야 할 때는 MCP 표준 엔드포인트를 사용하여 Gateway와 통신합니다.

Gateway 구현은 (MCP Authorization 사양)[https://modelcontextprotocol.org/specification/draft/basic/authorization]을 엄격하게 준수하여 강력한 보안과 액세스 제어를 보장합니다. 즉, Strands Agent가 도구를 호출할 때마다 권한 부여 단계를 거치므로 강력한 기능을 제공하면서 보안을 유지할 수 있습니다.

예를 들어 Strands Agent가 MCP 도구에 액세스해야 할 때 먼저 ListTools를 호출하여 사용 가능한 도구를 검색한 다음 InvokeTools를 사용하여 특정 작업을 실행합니다. Gateway는 필요한 모든 보안 검증, 프로토콜 변환 및 서비스 상호 작용을 처리하므로 전체 과정이 원활하고 안전하게 진행됩니다.

이 아키텍처 방식에서는 MCP 사양을 구현하는 모든 클라이언트 또는 SDK가 Gateway를 통해 AWS 서비스와 상호 작용할 수 있으므로 AI 에이전트 통합을 위한 다용도의 미래 지향적 솔루션을 제공합니다.

# 인바운드 권한 부여를 위해 Amazon Cognito에서 액세스 토큰 요청

In [ ]:
print(
    "Requesting the access token from Amazon Cognito authorizer...May fail for some time till the domain name propogation completes"
)
token_response = utils.get_token(user_pool_id, client_id, client_secret, scopeString, REGION)
token = token_response["access_token"]
print("Token response:", token)

# IT 에이전트에게 AWS 계정의 S3 리소스에 관해 질문

In [ ]:
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent


def create_streamable_http_transport():
    return streamablehttp_client(gatewayURL, headers={"Authorization": f"Bearer {token}"})


client = MCPClient(create_streamable_http_transport)

## ~/.aws/credentials에 구성된 IAM 그룹/사용자에게 Bedrock 모델 액세스 권한이 있어야 함
yourmodel = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    temperature=0.7,
)

In [ ]:
import logging


# 루트 strands 로거 구성. 문제를 디버깅하는 경우 DEBUG로 변경
logging.getLogger("strands").setLevel(logging.INFO)

# 로그를 확인할 수 있도록 핸들러 추가
logging.basicConfig(format="%(levelname)s | %(name)s | %(message)s", handlers=[logging.StreamHandler()])

with client:
    # listTools 호출
    tools = client.list_tools_sync()
    # 모델과 도구를 사용하여 Agent 생성
    agent = Agent(model=yourmodel, tools=tools)  ## 원하는 모델로 교체 가능
    # print(f"Tools loaded in the agent are {agent.tool_names}")
    # print(f"Tools configuration in the agent are {agent.tool_config}")
    # 샘플 프롬프트로 에이전트 호출. MCP listTools만 호출하여 LLM이 액세스할 수 있는 도구 목록을 조회하며, 아래에서는 실제 도구를 호출하지 않음
    agent("Hi , can you list all the s3 buckets. Locate the right tool and call the tool to get the answer")
    # agent("What is the weather in northern part of the mars")
    # 샘플 프롬프트로 에이전트와 도구를 호출하고 응답 표시
    # agent("Check the order status for order id 123 and show me the exact response from the tool")
    # MCP 도구를 명시적으로 호출. MCP 도구 이름과 인수는 AWS Lambda 함수 또는 OpenAPI/Smithy API와 일치해야 함
    result = client.call_tool_sync(
        tool_use_id="get-insight-weather-1",  # 고유 식별자로 교체 가능
        name=targetname
        + "___ListBuckets",  # AWS Lambda 대상 유형을 기반으로 한 도구 이름이며 대상 이름에 따라 변경됨
        # arguments={"ver": "1.0","feedtype": "json"}
    )
    # MCP 도구 응답 출력
    print(f"Tool Call result: {result['content'][0]['text']}")

# 정리
IAM 역할, IAM 정책, 자격 증명 공급자, AWS Lambda 함수, Cognito 사용자 풀, S3 버킷과 같은 추가 리소스도 생성됩니다. 정리 과정에서 이러한 리소스를 수동으로 삭제해야 할 수 있으며, 이는 실행한 예제에 따라 달라집니다.

## Gateway 삭제(선택 사항)

In [ ]:
import utils

utils.delete_gateway(gateway_client, gatewayID)